# Exploration Copilot - run everything from this notebook
Run the cells **top to bottom** (Shift+Enter). You do not need the terminal or Streamlit.

**Before you start**, put this notebook in the same folder as `ingest.py`, `tools.py` and `agent.py`.
Your data stays where it is - you only give the notebook the path to the data folder in Step 2 (nothing is copied).
The data folder is searched recursively, so sub-folder names do not matter much. Expected content:
```
challenge1_challenge2/
  inversion/   4 .sgy
  stack/       5 .sgy
  well_data/   WellList_Information.xlsx, deviations/*.txt (13), checkshot/*.txt (3), las/*.las (13)
```

## Step 1 - Install packages (only needed once)

In [ ]:
%pip install -q numpy pandas openpyxl scipy matplotlib segyio lasio openai scikit-image pyshp   # torch is provided by the organisers

## Step 2 - Settings
If you get `No module named 'ingest'`, set `PROJECT_DIR` to the folder that holds `ingest.py`. `DATA_DIR` is the full path to your data folder; the files are read from there and never copied. If you change these later, restart the kernel (Kernel > Restart) and run again.

In [ ]:
import os, sys
# Folder that contains ingest.py, tools.py, agent.py and the knowledge/ folder.
# Leave "" if this notebook is already in that folder; otherwise give the full path.
PROJECT_DIR = ""     # e.g. "/home/you/copilot"

if PROJECT_DIR:
    os.chdir(PROJECT_DIR)
sys.path.insert(0, os.getcwd())
print("Working folder:", os.getcwd())
print("Project files found:", [f for f in ("ingest.py", "tools.py", "agent.py") if os.path.exists(f)])

DATA_DIR = "/data/epic20/APGCE/input_data/challenge1_challenge2"   # RAW data: folder that contains inversion/, stack/, well_data/
INTERP_DIR = "/data/epic20/APGCE/apgce_team_11/insight-geohackathon/data_export"   # INTERPRETED data: Fault (SHP)/, Polygon (SHP)/, Horizon/  ("" = none)
CACHE_DIR = "cache"                         # where the processed data cards are saved
LLM_BASE_URL = "http://172.28.34.1:8000/v1"  # shared Gemma server

os.environ["DATA_DIR"] = DATA_DIR
os.environ["CACHE_DIR"] = CACHE_DIR
os.environ["INTERP_DIR"] = INTERP_DIR
os.environ["LLM_BASE_URL"] = LLM_BASE_URL
print("Settings saved")

## Step 3 - Check your files are found
You should see 1 Excel, 13 LAS, 9 SEG-Y and 16 text files (13 deviation + 3 checkshot).

In [ ]:
from pathlib import Path
d = Path(DATA_DIR)
print("Data folder exists:", d.exists(), "->", d.resolve())
for label, pattern in [("Excel", "*.xls*"), ("LAS", "*.las"), ("SEG-Y", "*.sgy"), ("SEG-Y (.segy)", "*.segy"), ("Text (deviation/checkshot)", "*.txt")]:
    files = sorted(d.rglob(pattern))
    print(f"{label:28s}: {len(files)} files")
for f in sorted(d.rglob("*.sgy")):
    print("   ", f.relative_to(d))

## Step 4 - Test the connection to Gemma
If this fails you are probably not on the hackathon network. The data steps below still work without it.

In [ ]:
import openai
client = openai.OpenAI(base_url=LLM_BASE_URL, api_key="token-not-needed")
try:
    r = client.chat.completions.create(
        model="google/gemma-4-26B-A4B-it",
        messages=[{"role": "user", "content": "Say hello in one short sentence."}],
        max_tokens=50)
    print("Gemma replied:", r.choices[0].message.content)
except Exception as e:
    print("Could not reach Gemma:", e)

## Step 5 - Read all the data (takes a few minutes)
This reads the Excel, deviations, checkshots, LAS logs and SEG-Y volumes, and saves the results into `cache/`. Read the **warnings** at the end - they tell you what to fix.

In [ ]:
import ingest
cards = ingest.main()

## Step 6 - Look at what was built

In [ ]:
import pandas as pd
import tools
pd.set_option("display.width", 200)
pd.DataFrame(tools.list_wells())

In [ ]:
pd.DataFrame(tools.get_volume_info())

In [ ]:
print("WARNINGS:")
for w in cards["warnings"] or ["none"]:
    print(" -", w)
print("\nWells inside each inversion volume (nearest trace distance in map units):")
rows = []
for w, c in cards["wells"].items():
    row = {"well": w}
    for v, cov in c["volume_coverage"].items():
        if cards["volumes"][v]["kind"] == "inversion":
            row[v] = "inside" if cov["inside"] else f"OUTSIDE ({cov['nearest_trace_m']:.0f})"
    rows.append(row)
pd.DataFrame(rows)

## Step 6 - Diagnose the interpreted data (paste this output if something looks wrong; it prints ranges only, not data)

In [ ]:
import interp_data, json
interp_data.diagnose()
print(json.dumps(tools.list_datasets()["extents_xy"], indent=1))

## Step 6a - How was the Fault (CSV) folder understood? (only if you use CSV faults)

In [ ]:
import fault_csv, os
fault_csv.describe(os.path.join(INTERP_DIR, "Fault (CSV)"))

## Step 6b - What interpreted data was found?

In [ ]:
inv = tools.list_datasets()
inv["interpreted"] if isinstance(inv["interpreted"], str) else (inv["interpreted"]["summary"], inv["interpreted"]["warnings"])

## Step 7 - Try a tool directly (no LLM needed)

In [ ]:
tools.get_well_summary("eo-1")

In [ ]:
tools.rank_wells("sand probability")

In [ ]:
tools.crosscheck_well("no-1")

## Step 8 - Look at a seismic picture
Change the volume / `kind` (`inline`, `xline`, `time`) / `index` (line number, or TWT in ms for `time`). Wells are marked in red.

In [ ]:
from IPython.display import Image, display, Markdown
v = tools.get_volume_info("sand probability")
mid_ms = (v["time_start_ms"] + v["time_end_ms"]) / 2
res = tools.render_slice("sand probability", "time", mid_ms)
display(Image(res["image_path"]))

## Step 9 - Chat with the copilot
Run the cell below once to define `ask()`, then type any question inside `ask("...")`. The list under each answer shows which tools Gemma used.

In [ ]:
import agent
from IPython.display import Image, display, Markdown

history = []

def ask(question, show_tools=True):
    out = agent.run(history, question)
    history.extend([{"role": "user", "content": question},
                    {"role": "assistant", "content": out["answer"]}])
    del history[:-10]
    display(Markdown(out["answer"]))
    for p in out["images"]:
        display(Image(p))
    if show_tools:
        print("\nTools used:")
        for t in out["trace"]:
            print(f"  {t['round']}. {t['tool']}({t['args']})")

print("Ready - use ask('your question')")

In [ ]:
ask("Give me an overview of the dataset: wells, volumes and data quality caveats.")

In [ ]:
ask("Which wells show the highest sand probability, and do their GR logs agree?")

In [ ]:
ask("Based on the seismic and well data, is this prospect more likely a structural or a stratigraphic trap?")

In [ ]:
ask("Why do you think that, step by step? What evidence is missing?")

In [ ]:
ask("Compare wells no-1 and o-1.")

In [ ]:
ask("What are the key geological risks that should be investigated next?")

## AI interpretation (PyTorch)
**Faults:** a 3D U-Net trained on synthetic seismic (no field labels needed). `models/fault_unet.pt` is a quick 300-step version; retrain for better quality (about 20-30 min on a GPU).
**Facies:** unsupervised k-means classes, no training needed.

In [ ]:
# OPTIONAL: (re)train the fault model. Use 1500+ steps on a GPU. Weights are saved to models/fault_unet.pt
import ai_interp
# ai_interp.train(steps=1500, batch=4)

In [ ]:
import ai_interp; print(ai_interp.available()); ai_interp.selftest(4)

In [ ]:
r = tools.ai_fault_detection()
if 'error' in r:
    print(r)
else:
    display(Image(r['image_path']))
    print(r['method'], '| faults:', r.get('n_faults', r.get('n_fault_candidates')))
    pd.DataFrame(r.get('faults', []))

In [ ]:
r = tools.ai_facies_map()
if 'error' in r:
    print(r)
else:
    display(Image(r['image_path']))
    print(r['class_at_wells'])
    pd.DataFrame(r['class_statistics'])

## Direct tests (no LLM): these call the tools straight away

In [ ]:
r = tools.high_variation_slices(5)
if 'error' in r:
    print(r)
else:
    display(Image(r['image_path']))
    pd.DataFrame(r['top_slices'])

In [ ]:
r = tools.depositional_pattern()
if 'error' in r:
    print(r)
else:
    display(Image(r['image_path']))
    {k: r[k] for k in ('window_ms','sand_fraction','n_bodies','proposed_environment','because','confidence')}

In [ ]:
r = tools.count_faults()
if 'error' in r:
    print(r)
else:
    display(Image(r['image_path']))
    print('Fault candidates:', r['n_fault_candidates'])
    pd.DataFrame(r['faults'])

## Interpreted-data tools (no LLM)

In [ ]:
r = tools.interpreted_faults()
if 'error' in r:
    print(r)
else:
    display(Image(r['image_path']))
    print(r['n_unique_faults'], 'unique faults;', r['linking_method'])
    display(pd.DataFrame(r['faults']))

In [ ]:
r = tools.interpreted_facies()
if 'error' in r:
    print(r)
else:
    display(Image(r['image_path']))
    print(r['interpretation'])

In [ ]:
r = tools.horizon_structure()
if 'error' in r:
    print(r)
else:
    display(Image(r['image_path']))
    display(pd.DataFrame(r['closures']))

In [ ]:
r = tools.assess_traps()
if 'error' in r:
    print(r)
else:
    display(Image(r['image_path']))
    print({k: (v['status'], v['score']) for k, v in r['trap_types'].items()})
    print(r['leading_assessment'], r['confidence'])

In [ ]:
r = tools.rank_prospects()
if 'error' in r:
    print(r)
else:
    display(pd.DataFrame(r['prospects']))

In [ ]:
tools.validate_interpreted_vs_seismic()

## Add new data later
Copy new shapefiles / horizon files into the interpreted folder (or new raw files into the raw folder), then run this cell. Only new or changed files are read, and every tool and question then uses them.

In [ ]:
ingest_all.refresh()

## Suggested answer + reasoning (trap identification)

In [ ]:
ask("Identify whether the trap is stratigraphic, structural or combined in the window 2000-2300 ms. Give your suggested answer and the reasoning from the data.")

In [ ]:
r = tools.recommend_trap(2000, 2300)
print(r['suggested_answer'], '| confidence:', r['confidence'])
pd.DataFrame(r['reasoning_chain'])

## Maths on volumes and gas potential (no LLM)

In [ ]:
r = tools.volume_math('far - near', 2000, 2300, select='low', percent=10)
print(r.get('error') or r['stats'])
display(Image(r['image_path'])) if 'image_path' in r else None
pd.DataFrame(r.get('zones', []))

In [ ]:
r = tools.gas_potential(2000, 2300)
print(r.get('error') or r['answer_markdown'])
display(Image(r['image_path'])) if 'image_path' in r else None

In [ ]:
ask("Where is the probable gas potential location in the window 2000-2300 ms? Use far minus near with high negative values, structurally high and high sand probability.")

In [ ]:
ask("Calculate far minus near in the window 2000-2300 ms and show where it is most negative.")

## Interpretation questions

In [ ]:
ask("Find the time slice with the highest variation of sand probability (0 to 1) and make the map.")

In [ ]:
ask("Use the sand probability volume to define the most likely depositional environment pattern.")

In [ ]:
ask("How many faults are detected?")

In [ ]:
ask("Classify the seismic facies around the most interesting time slice and relate the classes to the wells.")

In [ ]:
ask("Based on the seismic and well data, what is the most likely trapping mechanism?")

In [ ]:
ask("Where is the most likely prospect? Give inline, crossline and time slice.")

In [ ]:
ask("Convert the well logs to the time domain, select the thickest gas zone in any well, compare it with the seismic in time, and extrapolate it as a map away from the well.")

In [ ]:
ask("Use the sand probability volume to define the most likely depositional environment pattern.")

## Your own question
Edit the text and run.

In [ ]:
ask("")

### Optional: web app version
Open a terminal in this folder and run `streamlit run app.py` (after `pip install streamlit`). Everything above already works without it.